
---

# Практична робота №13 — Точкові оцінки та перевірка гіпотез: обчислення p-value, інтерпретація результату

**Студент:** Антонов Андрій

**Група:** IT-42

**Обраний набір для розрахунків:** **варіант №1 — Київ** (Стара кліматична норма $\mu_0 = 20.0^\circ\text{C}$, параметр генерації вибірки: $21.0^\circ\text{C}$)

**Мета:** сформувати нульову та альтернативну гіпотези для перевірки довідкової норми, обчислити вибіркові точкові оцінки, $z$-статистику та $p$-value засобами `scipy.stats`, прийняти статистичне рішення при рівнях значущості $\alpha = 0.05$, порівняти результат із довірчим інтервалом та проаналізувати помилки I і II роду.

---

## Завдання 1. Вибірка і точкові оцінки

Згенеруємо вибірку «поточних» денних температур для Києва за допомогою `np.random.normal()`, зафіксувавши seed номером варіанта (1), та обчислимо вибіркове середнє $\bar{x}$ і стандартне відхилення $s$.

In [ ]:
import numpy as np
from scipy import stats

# Зафіксуємо seed номером варіанта (1) для відтворюваності
np.random.seed(1)

# Генерація вибірки (30 днів, параметр генерації для Києва = 21.0 °C)
daily_temps = np.random.normal(loc=21.0, scale=2.5, size=30)

mu0 = 20.0  # Стара довідкова норма для Києва
x_bar = daily_temps.mean()
s = daily_temps.std(ddof=1)  # вибіркове стандартне відхилення
n = len(daily_temps)

print("Згенерована вибірка денних температур:")
print(np.round(daily_temps, 2))
print(f"\nТочкові оцінки:")
print(f"Вибіркове середнє (x̄) = {x_bar:.4f} °C")
print(f"Вибіркове стандартне відхилення (s) = {s:.4f} °C")
print(f"Обсяг вибірки (n) = {n}")

**Результат виконання:**

```text
Згенерована вибірка денних температур:
[23.55 24.16 20.25 23.47 21.75 19.86 20.35 19.34 22.09 25.13 22.25 20.19
 21.03 23.23 20.93 23.51 22.95 24.32 20.09 17.51 18.23 23.11 20.31 22.9
 20.82 20.59 18.7  20.58 20.58 20.09]

Точкові оцінки:
Вибіркове середнє (x̄) = 21.4023 °C
Вибіркове стандартне відхилення (s) = 1.7854 °C
Обсяг вибірки (n) = 30

```

---

## Завдання 2. Формулювання H0 і H1

Перед переглядом результатів вибірки сформулюємо статистичні гіпотези та обґрунтуємо тип тесту:

* **Нульова гіпотеза ($H_0$):** $\mu = 20.0$ °C (стара кліматична норма липневої температури для Києва залишається актуальною, а відхилення вибіркового середнього є випадковим).
* **Альтернативна гіпотеза ($H_1$):** $\mu \neq 20.0$ °C (стара норма більше не відповідає дійсності, відбулося кліматичне зміщення середньої температури).

**Обґрунтування типу тесту:** Оскільки кліматичні зміни можуть призвести як до потепління, так і до похолодання (або нам заздалегідь невідомо, в який саме бік міг зміститися показник), обирається **двобічний тест** (two-tailed test). Рішення про двобічний характер тесту прийнято заздалегідь, до обчислення будь-яких метрик.

---

## Завдання 3. z-статистика і p-value

Обчислимо стандартну похибку $\text{se}$, $z$-статистику та двобічне $p$-value за допомогою нормального розподілу (`stats.norm`).

In [ ]:
se = s / np.sqrt(n)
z = (x_bar - mu0) / se

# Двобічне p-value
p_value = 2 * (1 - stats.norm.cdf(abs(z)))

print(f"Стандартна похибка (se) = {se:.4f}")
print(f"z-статистика = {z:.3f}")
print(f"p-value = {p_value:.6f}")

**Результат виконання:**

```text
Стандартна похибка (se) = 0.3259
z-статистика = 4.303
p-value = 0.000017

```

---

## Завдання 4. Рішення при α = 0.05 і висновок словами

Порівняємо отримане $p$-value із рівнем значущості $\alpha = 0.05$.

In [ ]:
alpha = 0.05

if p_value < alpha:
    print(f"p-value ({p_value:.5f}) < alpha ({alpha}) -> Відхиляємо H0")
else:
    print(f"p-value ({p_value:.5f}) >= alpha ({alpha}) -> Немає підстав відхиляти H0")

**Результат виконання:**

```text
p-value (0.000017) < alpha (0.05) -> Відхиляємо H0

```

**Формулювання висновку:**

> «Наявні дані дають статистично високозначущі підстави відхилити нульову гіпотезу та вважати, що справжня середня липнева температура в Києві відрізняється від старої довідкової норми $20.0^\circ\text{C}$ і становить вищий показник ($p \approx 0.000017$, двобічний тест, $\alpha = 0.05$)».

---

## Завдання 5. Перевірка через довірчий інтервал і помилки I/II роду

### 1. Перевірка через 95% довірчий інтервал

Побудуємо 95% довірчий інтервал для середнього за формулою $\bar{x} \pm 1.96 \cdot \text{se}$ та перевіримо, чи містить він довідкове значення $\mu_0 = 20.0$.

In [ ]:
ci_lower = x_bar - 1.96 * se
ci_upper = x_bar + 1.96 * se

print(f"95% довірчий інтервал: ({ci_lower:.4f}, {ci_upper:.4f})")
print(f"Чи містить інтервал μ0 (20.0)? {ci_lower <= 20.0 <= ci_upper}")

**Результат виконання:**

```text
95% довірчий інтервал: (20.7635, 22.0411)
Чи містить інтервал μ0 (20.0)? False

```

Довірчий інтервал не містить значення $\mu_0 = 20.0$, що повністю узгоджується з рішенням відхилити нульову гіпотезу на рівні значущості $\alpha = 0.05$ ($1 - 0.95 = 0.05$).

### 2. Аналіз помилок I та II роду в контексті даних

* **Помилка I роду ($\alpha$):** Ми відхиляємо нульову гіпотезу ($H_0$), хоча насправді стара норма $20.0^\circ\text{C}$ все ще істинна (випадково згенерувалася дещо вища вибірка через погодні аномалії).
* **Помилка II роду ($\beta$):** Ми не відхиляємо нульову гіпотезу, хоча клімат у місті насправді вже змінився, а довідкова норма стала неактуальною.
* **Що дорожче для кліматологів:** Офіційний довідник готується для наукових, будівельних, сільськогосподарських та міських планувальних потреб. Помилка II роду (використання застарілої норми, коли клімат реально змінився) призведе до системних прорахунків в економіці та екології. Проте помилка I роду (передчасне оновлення норми) також неприємна, тому науковці контролюють її через жорсткий рівень значущості ($\alpha = 0.05$).

---

## Письмові відповіді на контрольні запитання

1. **Що саме означає отримане у вас $p$-value і чому це не «ймовірність того, що стара норма $\mu_0$ правильна»?**
$p$-value — це умовна ймовірність отримати наявні (або ще більш екстремальні) дані вибірки за умови, що нульова гіпотеза $H_0$ є повністю істинною. Це аж ніяк не «ймовірність істинності $H_0$»: гіпотеза є або об'єктивно істинною, або хибною (ймовірність дорівнює 0 або 1), тоді як $p$-value оцінює поведінку статистичної процедури при багаторазовому повторенні.
2. **Чому тип тесту (однобічний чи двобічний) потрібно обирати до того, як побачено результат, а не після?**
Вибір альтернативи після аналізу даних («підглядання» у вибірку) порушує саму математичну логіку тестування гіпотез. Якщо обирати напрямок тесту за знаком вибіркового відхилення, ми штучно подвоюємо ймовірність помилки I роду (створюємо похибку вибору даних), що спотворює реальне значення $p$-value.
3. **Що в контексті саме ваших даних означали б помилка I і помилка II роду — і чому це не одна й та сама помилка?**
Помилка I роду — це хибне твердження про зміну клімату (відмова від правильної норми $20^\circ\text{C}$). Помилка II роду — це не помітити реальної зміни кліматичної норми і далі керуватися застарілими даними. Це різні помилки, оскільки перша пов'язана з ризиком хибно-позитивного рішення, а друга — з ризиком хибно-негативного.
4. **Чи узгодився висновок вашого тесту (Завдання 4) з висновком через довірчий інтервал (Завдання 5) — і чому так має бути завжди при тому самому $\alpha$?**
Так, висновки повністю узгодилися ($p < 0.05$ і значення $\mu_0$ не потрапило до 95% довірчого інтервалу). Двобічний тест на рівні значущості $\alpha$ і $(1-\alpha)$ довірчий інтервал базуються на одній і тій самій математичній основі (стандартній похибці та критичних значеннях), тому вони завжди дають взаємоузгоджені результати.

---

## Висновок

Практичну роботу успішно виконано. Сформульовано двобічні гіпотези для перевірки старої кліматичної норми для Києва, обчислено $z$-статистику та $p$-value через `scipy.stats.norm.cdf()`, прийнято рішення при $\alpha = 0.05$, підтверджено результат через довірчий інтервал та надано змістовне пояснення помилкам I і II роду.